# OceanEye — Perception Stage (Stages 1–4)
**Pipeline:** Video in → YOLO person detection → ByteTrack tracking → overlay with track IDs & trajectories.

**100% Local & Self-Contained:**
- Code: loaded directly from `C:\pbl_oceaneye\app`
- Videos: loaded directly from `C:\pbl_oceaneye\videos`
- Outputs: written directly to `outputs\perception\<run_name>\` (`overlay.mp4`, `tracks.jsonl`, `summary.json`)
- No Google Drive, no external cloud dependencies, no upload prompts.

## 1 · Environment & GPU Check

In [1]:
#@title 1 · Environment & GPU Check { display-mode: "form" }
import importlib.util, os, platform, subprocess, sys

print("Python:", sys.version.split()[0], "|", platform.platform())
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10
    ).stdout.strip()
    print("GPU:", gpu or "None detected (CPU will be used)")
except Exception:
    print("GPU: None detected (CPU will be used)")

Python: 3.10.7 | Windows-10-10.0.26300-SP0
GPU: NVIDIA GeForce RTX 2050, 4096 MiB


## 2 · Load the OceanEye Codebase

In [1]:
#@title 2 · Load the OceanEye codebase { display-mode: "form" }
PROJECT_DIR = r"C:\pbl_oceaneye"  #@param {type:"string"}

import os, sys, pathlib
from pathlib import Path

def find_project_root() -> Path:
    if PROJECT_DIR.strip():
        p = Path(PROJECT_DIR.strip())
        if (p / "app").is_dir() and (p / "pyproject.toml").is_file():
            return p.resolve()
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "app").is_dir() and (p / "pyproject.toml").is_file():
            return p.resolve()
    default_p = Path(r"C:\pbl_oceaneye")
    if (default_p / "app").is_dir():
        return default_p.resolve()
    return None

ROOT = find_project_root()
assert ROOT is not None, f"Could not find OceanEye project root containing 'app/'. Checked {PROJECT_DIR} and {Path.cwd()}"

for mod in [m for m in sys.modules if m == "app" or m.startswith("app.")]:
    del sys.modules[mod]

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)
print("Project root:", ROOT)
print("Working directory:", os.getcwd())

Project root: C:\pbl_oceaneye
Working directory: C:\pbl_oceaneye


## 3 · Install Vision Dependencies

In [2]:
#@title 3 · Install vision dependencies { display-mode: "form" }
import importlib.util, subprocess, sys

needed = {
    "ultralytics": "ultralytics",
    "lap": "lap",
    "cv2": "opencv-python",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing missing dependencies: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required vision packages are already installed.")

import cv2, torch, ultralytics
print(f"ultralytics: {ultralytics.__version__} | torch: {torch.__version__} | CUDA: {torch.cuda.is_available()} | OpenCV: {cv2.__version__}")

All required vision packages are already installed.
ultralytics: 8.4.171 | torch: 2.14.1+cpu | CUDA: False | OpenCV: 5.0.0


## 4 · Choose Input Video
Scans `C:\pbl_oceaneye\videos` directly. Choose a video by index or toggle `PROCESS_ALL = True`.

In [3]:
#@title 4 · Choose input video { display-mode: "form" }
VIDEO_SELECTION = "gettyimages-89242358-640_adpp.mp4"  #@param ["gettyimages-89242358-640_adpp.mp4", "gettyimages-2214078267-640_adpp.mp4", "gettyimages-2297913922-640_adpp.mp4", "ALL_VIDEOS"]
VIDEOS_DIR_PATH = r"C:\pbl_oceaneye\videos"  #@param {type:"string"}
CUSTOM_VIDEO_PATH = ""  #@param {type:"string"}

import os, sys, pathlib, cv2
from pathlib import Path

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".m4v", ".webm"}

# Purge any leftover fake demo video files
for p in [ROOT / "videos", Path("/content/oceaneye/videos"), Path("/content/videos"), Path("/content")]:
    if p.is_dir():
        for d in p.glob("demo_*"):
            try: d.unlink()
            except Exception: pass

def find_videos_dir() -> Path:
    candidates = [
        Path(VIDEOS_DIR_PATH.strip()) if VIDEOS_DIR_PATH.strip() else None,
        ROOT / "videos",
        Path(r"C:\pbl_oceaneye\videos"),
        Path.cwd() / "videos",
        Path("/content/oceaneye/videos"),
        Path("/content/videos"),
        Path("/content"),
    ]
    for c in candidates:
        if c and c.is_dir():
            files = [p for p in c.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTS and not p.name.startswith("demo_") and not p.name.startswith("overlay_")]
            if files:
                return c.resolve()
    return Path(VIDEOS_DIR_PATH.strip()) if VIDEOS_DIR_PATH.strip() else ROOT / "videos"

VIDEOS_DIR = find_videos_dir()
AVAILABLE_VIDEOS = sorted([p for p in VIDEOS_DIR.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTS and not p.name.startswith("demo_") and not p.name.startswith("overlay_")]) if VIDEOS_DIR.is_dir() else []

if CUSTOM_VIDEO_PATH.strip():
    custom_p = Path(CUSTOM_VIDEO_PATH.strip()).resolve()
    assert custom_p.is_file(), f"Custom video file not found: {custom_p}"
    SELECTED_VIDEOS = [custom_p]
    VIDEO_PATH = str(custom_p)
elif VIDEO_SELECTION == "ALL_VIDEOS":
    assert AVAILABLE_VIDEOS, f"No video files found in {VIDEOS_DIR}."
    SELECTED_VIDEOS = AVAILABLE_VIDEOS
    VIDEO_PATH = str(AVAILABLE_VIDEOS[0])
else:
    assert AVAILABLE_VIDEOS, (
        f"No real video files found in {VIDEOS_DIR}!\n\n"
        "HOW TO FIX:\n"
        "• IF ON COLAB: Click the Folder (Files) icon on Colab's left sidebar, and drag 'gettyimages-89242358-640_adpp.mp4' from 'C:\\pbl_oceaneye\\videos' into '/content/' or '/content/oceaneye/videos/'.\n"
        "• IF IN VS CODE: In the top-right corner of this notebook, switch the kernel from Colab to your local 'venv' — it will run on your RTX 2050 GPU using C:\\pbl_oceaneye\\videos directly without any upload!"
    )
    match = [p for p in AVAILABLE_VIDEOS if p.name == VIDEO_SELECTION]
    chosen = match[0] if match else AVAILABLE_VIDEOS[0]
    SELECTED_VIDEOS = [chosen]
    VIDEO_PATH = str(chosen)

print(f"Videos folder: {VIDEOS_DIR}")
print(f"Active video for processing: {VIDEO_PATH}")
print(f"Selected queue: {[p.name for p in SELECTED_VIDEOS]}\n")
print(f"Available videos ({len(AVAILABLE_VIDEOS)} found):")
for i, v in enumerate(AVAILABLE_VIDEOS):
    cap = cv2.VideoCapture(str(v))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    fps = cap.get(cv2.CAP_PROP_FPS) or 0
    cnt = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    dur = cnt / fps if fps > 0 else 0
    cap.release()
    marker = " <--- [ACTIVE]" if v in SELECTED_VIDEOS else ""
    print(f"  [{i}] {v.name}: {w}x{h}, {cnt} frames ({dur:.1f}s @ {fps:.1f} fps){marker}")

Videos folder: C:\pbl_oceaneye\videos
Active video for processing: C:\pbl_oceaneye\videos\gettyimages-89242358-640_adpp.mp4
Selected queue: ['gettyimages-89242358-640_adpp.mp4']

Available videos (3 found):
  [0] gettyimages-2214078267-640_adpp.mp4: 768x432, 285 frames (9.5s @ 30.0 fps)
  [1] gettyimages-2297913922-640_adpp.mp4: 768x432, 1062 frames (17.7s @ 59.9 fps)
  [2] gettyimages-89242358-640_adpp.mp4: 768x432, 638 frames (26.6s @ 24.0 fps) <--- [ACTIVE]


## 5 · Perception Pipeline Settings
Settings tuned for swimmer perception:
- `MODEL`: `yolov8s.pt` (good balance for small swimmers in waves)
- `IMGSZ`: 960 (larger image size resolves distant swimmers)
- `CONF`: 0.10 (low on purpose so ByteTrack maintains tracks through wave occlusions)
- `TRAIL`: 60 frames trajectory tail

In [4]:
#@title 5 · Perception Pipeline Settings { display-mode: "form" }
MODEL = "yolov8s.pt"  #@param ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt", "yolov8l.pt", "yolov8x.pt"]
DEVICE = "auto"  #@param ["auto", "cpu", "cuda:0"]
IMGSZ = 960  #@param {type:"integer"}
CONF = 0.10  #@param {type:"number"}
STRIDE = 1  #@param {type:"integer"}
MAX_FRAMES = 0  #@param {type:"integer"}
TRAIL = 60  #@param {type:"integer"}
CAMERA_ID = "cam_01"  #@param {type:"string"}

from app.vision.config import DetectorConfig, OverlayConfig, PerceptionConfig, TrackerConfig

cfg = PerceptionConfig(
    camera_id=CAMERA_ID,
    detector=DetectorConfig(
        model=MODEL,
        device=DEVICE,
        imgsz=IMGSZ,
        conf=CONF,
        weights_dir=str(ROOT / "models"),
    ),
    tracker=TrackerConfig(),
    overlay=OverlayConfig(trail_length=TRAIL),
    stride=STRIDE,
    max_frames=MAX_FRAMES or None,
    output_dir=str(ROOT / "outputs" / "perception"),
)

print("Perception configuration ready:")
print(f"  Camera: {cfg.camera_id} | Model: {MODEL} | Device: {DEVICE} | ImgSz: {IMGSZ} | Conf: {CONF}")
print(f"  Weights folder: {cfg.detector.weights_dir}")
print(f"  Outputs folder: {cfg.output_dir}")
print(f"  Queue: {[p.name for p in SELECTED_VIDEOS]}")

Perception configuration ready:
  Camera: cam_01 | Model: yolov8s.pt | Device: auto | ImgSz: 960 | Conf: 0.1
  Weights folder: C:\pbl_oceaneye\models
  Outputs folder: C:\pbl_oceaneye\outputs\perception
  Queue: ['gettyimages-89242358-640_adpp.mp4']


## 6 · Run Detection + Tracking
Runs YOLO + ByteTrack and draws trajectories with live preview.

In [5]:
#@title 6 · Run detection + tracking { display-mode: "form" }
PREVIEW_INTERVAL = 40  #@param {type:"integer"}

import cv2, json
import pandas as pd
from IPython.display import Image, clear_output, display
from app.vision.runner import PerceptionRunner

def _preview(frame, result, annotated):
    if annotated is not None and frame.index % max(1, PREVIEW_INTERVAL) == 1:
        scale = min(1.0, 640 / annotated.shape[1])
        clear_output(wait=True)
        print(f"Processing {CURRENT}: frame {frame.index} | people tracked: {len(result.tracks)}")
        display(Image(data=cv2.imencode(".jpg", cv2.resize(annotated, None, fx=scale, fy=scale))[1].tobytes()))

SUMMARIES = []
for video in SELECTED_VIDEOS:
    CURRENT = video.name
    print(f"\n--- Starting processing: {CURRENT} ---")
    s = PerceptionRunner(cfg, on_frame=_preview).run(str(video))
    SUMMARIES.append(s)

clear_output(wait=True)
summary = SUMMARIES[0]

cols = [
    "source", "frames_processed", "source_fps", "pipeline_fps", "avg_pipeline_ms",
    "unique_track_ids", "max_concurrent_tracks", "device", "model"
]
display(pd.DataFrame([{c: getattr(s, c, "") for c in cols} for s in SUMMARIES]))

,source,frames_processed,source_fps,pipeline_fps,avg_pipeline_ms,unique_track_ids,max_concurrent_tracks,device,model
0,gettyimages-89242358-640_adpp,638,23.976,2.41,415.45,27,3,cpu,yolov8s.pt


## 7 · Watch the Result
Displays sample annotated frames and embedded video playback.

In [ ]:
#@title 7 · Watch the result { display-mode: "form" }
PLAY_VIDEO_INDEX = 0  #@param {type:"integer"}

import matplotlib.pyplot as plt
import subprocess, shutil
from pathlib import Path
from IPython.display import Video, display

if "SUMMARIES" not in globals() or not SUMMARIES:
    print("Please run Cell 6 first to generate pipeline outputs.")
else:
    for s in SUMMARIES:
        path = s.outputs.get("overlay_video")
        if not path or not Path(path).exists():
            continue
        cap = cv2.VideoCapture(str(path))
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
        fig, axes = plt.subplots(2, 3, figsize=(15, 6))
        fig.suptitle(f"Sample frames: {Path(s.source).name}")
        for ax, k in zip(axes.flat, range(6)):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(k * (n - 1) / 5))
            ok, img = cap.read()
            if ok:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.axis("off")
        cap.release()
        plt.tight_layout()
        plt.show()

    if PLAY_VIDEO_INDEX is not None and 0 <= PLAY_VIDEO_INDEX < len(SUMMARIES):
        src = SUMMARIES[PLAY_VIDEO_INDEX].outputs.get("overlay_video")
        if src and Path(src).exists():
            dst = str(Path(src).with_name("overlay_h264.mp4"))
            ok = shutil.which("ffmpeg") and subprocess.run(
                ["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-vcodec", "libx264", "-pix_fmt", "yuv420p", dst]
            ).returncode == 0
            display(Video(dst if ok else src, embed=True, width=720))

## 8 · Tracking Analytics & Trajectories

In [ ]:
#@title 8 · Tracking analytics & trajectories { display-mode: "form" }
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def load_tracks(s):
    rows = []
    p = s.outputs.get("tracks_jsonl")
    if not p or not Path(p).exists():
        return pd.DataFrame(rows)
    for line in open(p, encoding="utf-8"):
        rec = json.loads(line)
        for t in rec.get("tracks", []):
            x1, y1, x2, y2 = t["bbox"]
            rows.append(dict(
                frame=rec["frame_index"],
                t=rec["video_time_s"],
                id=t["track_id"].rsplit("-", 1)[-1],
                cx=(x1 + x2) / 2,
                cy=(y1 + y2) / 2,
                conf=t.get("conf", 0.0),
            ))
    return pd.DataFrame(rows)

if "SUMMARIES" not in globals() or not SUMMARIES:
    print("Please run Cell 6 first to generate pipeline outputs.")
else:
    for s in SUMMARIES:
        df = load_tracks(s)
        if df.empty:
            print(f"{Path(s.source).name}: No tracks recorded.")
            continue
        life = df.groupby("id").frame.count()
        print(f"{Path(s.source).name}: unique IDs {df.id.nunique()} | max concurrent {df.groupby('frame').id.nunique().max()} | short tracks (<1s): {(life < s.effective_fps).sum()}")
        fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))
        fig.suptitle(f"Tracking Analytics: {Path(s.source).name}")
        df.groupby("t").id.nunique().plot(ax=ax[0], title="People tracked over time")
        life.plot.hist(bins=20, ax=ax[1], title="Track lifetime (frames)")
        for tid, g in df.groupby("id"):
            ax[2].plot(g.cx, g.cy, lw=1)
        ax[2].invert_yaxis()
        ax[2].set_title("Trajectories (image coords)")
        plt.tight_layout()
        plt.show()

## 9 · (Optional) Speed Benchmark

In [ ]:
#@title 9 · Benchmark (Optional) { display-mode: "form" }
RUN_BENCHMARK = False  #@param {type:"boolean"}
BENCH_FRAMES = 60  #@param {type:"integer"}

import dataclasses
import pandas as pd

if RUN_BENCHMARK and "SELECTED_VIDEOS" in globals() and SELECTED_VIDEOS:
    rows = []
    test_video = str(SELECTED_VIDEOS[0])
    for model in ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]:
        for imgsz in [640, 960, 1280]:
            c = dataclasses.replace(
                cfg, max_frames=BENCH_FRAMES, save_video=False, save_tracks=False,
                output_dir=str(ROOT / "outputs" / "bench"),
                detector=dataclasses.replace(cfg.detector, model=model, imgsz=imgsz)
            )
            s = PerceptionRunner(c).run(test_video)
            rows.append(dict(
                model=model, imgsz=imgsz, ms_per_frame=round(s.avg_pipeline_ms, 2),
                fps=round(s.pipeline_fps, 1), unique_ids=s.unique_track_ids,
                max_concurrent=s.max_concurrent_tracks
            ))
    display(pd.DataFrame(rows))

## 10 · Verify Saved Outputs

In [ ]:
#@title 10 · Verify saved outputs { display-mode: "form" }
from pathlib import Path

if "SUMMARIES" not in globals() or not SUMMARIES:
    print("Please run Cell 6 first to generate pipeline outputs.")
else:
    for s in SUMMARIES:
        print(f"\nOutputs for {Path(s.source).name}:")
        rdir = Path(s.run_dir)
        print("  Directory:", rdir)
        for f in rdir.iterdir():
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"    - {f.name} ({size_mb:.2f} MB)")

## Next Steps (Roadmap)
* **Stage 5:** Manual danger-zone polygon per camera
* **Stage 6:** Zone breach & dwell-time detection
* **Stage 7:** Swimmer distress behaviour signals & rule-based risk scoring